<a href="https://colab.research.google.com/github/ZIZOUTOUMOU/LAB02-AdvDB/blob/main/tp_2_ml_for_iot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, glob, time, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score
from xgboost import XGBClassifier

# ---------------------------------------------------------------
# 0. Load the dataset
# ---------------------------------------------------------------
try:
    import kagglehub
    path = kagglehub.dataset_download("deepcontractor/smoke-detection-dataset")
    csv_path = glob.glob(f"{path}/**/*.csv", recursive=True)[0]
except Exception as e:
    print("Download failed:", e)
    print("Upload smoke_detection_iot.csv manually:")
    from google.colab import files
    up = files.upload()
    csv_path = list(up.keys())[0]

df = pd.read_csv(csv_path)
print("Loaded:", csv_path, df.shape)

# Drop index/time columns, keep the sensor features
drop_cols = [c for c in df.columns if c.lower().startswith("unnamed") or c.lower() == "utc"]
df = df.drop(columns=drop_cols)
df = df.dropna()

target_col = next(c for c in df.columns if "fire" in c.lower())
X = df.drop(columns=[target_col])
y = df[target_col]
print("Features:", X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------------
# 1. Pipelines
# ---------------------------------------------------------------
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

xgb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(eval_metric="logloss", random_state=42)),
])

pipelines = {"LR-Pipeline": lr_pipe, "XGB-Pipeline": xgb_pipe}

# ---------------------------------------------------------------
# Task 1: Train and evaluate (also time the training)
# ---------------------------------------------------------------
results = {}
for name, pipe in pipelines.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    y_pred = pipe.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Train time (s)": train_time,
    }

print("\n=== Task 1: Performance ===")
print(pd.DataFrame(results).T.round(4))

# ---------------------------------------------------------------
# Task 2.1: Model size (pickle)
# ---------------------------------------------------------------
sizes = {}
for name, pipe in pipelines.items():
    fname = f"{name}.pkl"
    with open(fname, "wb") as f:
        pickle.dump(pipe, f)
    sizes[name] = os.path.getsize(fname) / 1024  # KB

print("\n=== Task 2.1: Model size ===")
print(pd.DataFrame({"Size (KB)": sizes}).round(2))

# ---------------------------------------------------------------
# Task 2.2: Inference time
# ---------------------------------------------------------------
n_test = len(X_test)
timing = {}
for name, pipe in pipelines.items():
    pipe.predict(X_test.iloc[:10])  # warm-up so first-call overhead isn't counted
    runs = []
    for _ in range(5):              # repeat and average for a stable number
        t0 = time.perf_counter()
        pipe.predict(X_test)
        runs.append(time.perf_counter() - t0)
    total = float(np.mean(runs))
    timing[name] = {
        "Total Test Inference Time (s)": total,
        "Single Inference Time (ms)": total / n_test * 1000,
    }

print("\n=== Task 2.2: Inference time ===")
print(pd.DataFrame(timing).T.round(6))

# ---------------------------------------------------------------
# Extra: true single-sample latency (one row at a time)
# ---------------------------------------------------------------
print("\n=== Extra: real single-row latency (avg of 200 calls) ===")
for name, pipe in pipelines.items():
    t0 = time.perf_counter()
    for i in range(200):
        pipe.predict(X_test.iloc[[i]])
    print(f"{name}: {(time.perf_counter() - t0) / 200 * 1000:.3f} ms")

# ---------------------------------------------------------------
# Extra: raw parameter count (what actually has to go on the MCU)
# ---------------------------------------------------------------
lr = lr_pipe.named_steps["clf"]
n_lr_params = lr.coef_.size + lr.intercept_.size
n_trees = len(xgb_pipe.named_steps["clf"].get_booster().get_dump())
print(f"\nLR parameters: {n_lr_params} floats (~{n_lr_params * 4} bytes)")
print(f"XGB trees: {n_trees}")

100%|██████████| 1.64M/1.64M [00:00<00:00, 28.3MB/s]

Extracting files...


Loaded: /root/.cache/kagglehub/datasets/deepcontractor/smoke-detection-dataset/versions/1/smoke_detection_iot.csv (62630, 16)
Features: ['Temperature[C]', 'Humidity[%]', 'TVOC[ppb]', 'eCO2[ppm]', 'Raw H2', 'Raw Ethanol', 'Pressure[hPa]', 'PM1.0', 'PM2.5', 'NC0.5', 'NC1.0', 'NC2.5', 'CNT']

=== Task 1: Performance ===
              Accuracy      F1  Recall  Train time (s)
LR-Pipeline     0.9893  0.9925  0.9923          1.4138
XGB-Pipeline    1.0000  1.0000  1.0000          3.4332

=== Task 2.1: Model size ===
              Size (KB)
LR-Pipeline        1.69
XGB-Pipeline     116.76

=== Task 2.2: Inference time ===
              Total Test Inference Time (s)  Single Inference Time (ms)
LR-Pipeline                        0.010805                    0.000863
XGB-Pipeline                       0.038619                    0.003083

=== Extra: real single-row latency (avg of 200 calls) ===
LR-Pipeline: 5.199 ms
XGB-Pipeline: 12.620 ms

LR parameters: 14 floats (~56 bytes)
XGB trees: 100


##  Critical Deployment Analysis (ESP32)

**Memory feasibility.** Both models fit on an ESP32. The serialized LR-Pipeline is 1.69 KB and the XGB-Pipeline is 116.76 KB, both far below the several MB of Flash available. LR is tiny at runtime: 13 weights and one bias, plus the scaler's 13 means and 13 standard deviations, about 40 floats (roughly 160 bytes). XGBoost stores 100 trees and is about 70 times larger, so it is the more memory-constrained model and would take a meaningful share of the ~520 KB SRAM if loaded naively. Pickle files include Python object overhead, so an optimized C implementation would have a different size.

**Time efficiency.** On the test set, LR needs 0.00086 ms per sample and XGBoost 0.0031 ms (batch average on a desktop CPU). An ESP32 is much slower than a desktop, but even if it were several hundred times slower, both models would stay far below the 1-second budget: LR is a single dot product, and XGBoost is 100 shallow tree traversals. Both are therefore fast enough for one prediction per second. The single-row latency (about 5 ms for LR and 13 ms for XGBoost) is dominated by pandas/scikit-learn call overhead, not by the models themselves.

**Conclusion.** Logistic Regression is the clear choice for this application. It is about 70 times smaller and about 3.5 times faster than XGBoost, and it is simple enough to implement in a few lines of C/C++ (as done in TP3). XGBoost achieves slightly higher accuracy, but if it were required, it could be deployed by reducing `n_estimators` and `max_depth`, quantizing thresholds and leaf values to int8/int16, or exporting the trees to plain C code with tools such as `m2cgen` or `emlearn`.